In [ ]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q chromadb
!pip install -q groq
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [ ]:
import os

from google.colab import userdata

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

from groq import Groq

/tmp/ipykernel_982/2225672149.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
os.environ["GEMINI_KEY"] = userdata.get("GEMINI_KEY")
os.environ["GROQ_KEY"] = userdata.get("GROQ_KEY")

print("Keys loaded successfully")

Keys loaded successfully


In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_KEY")

print("Google API Key Loaded")

Google API Key Loaded


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Embedding model loaded")

Embedding model loaded


In [ ]:
from groq import Groq

client = Groq(
    api_key=os.environ["GROQ_KEY"]
)

print("Groq connected")

Groq connected


In [ ]:
def ingest_pdf(pdf_path):

    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    print("Pages loaded:", len(docs))

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(docs)

    print("Chunks created:", len(chunks))

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./chroma_db"
    )

    print("Vector database created")

    return vectorstore

In [ ]:
import os

print(os.listdir())

['.config', 'IEFT MODULE 1.pdf', 'sample_data']


In [ ]:
vectorstore = ingest_pdf("IEFT MODULE 1.pdf")

Pages loaded: 72
Chunks created: 77
Vector database created


In [ ]:
def ask_question(question, vectorstore):

    docs = vectorstore.similarity_search(
        question,
        k=3
    )

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
Use the provided context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
answer = ask_question(
    "What is Economics?",
    vectorstore
)

print(answer)

Economics is the study of how society and individuals use limited resources to satisfy their unlimited wants. It is also defined as a "science which studies human behaviour in relationship with given ends and scarce means" by Lionnel Robinson in 1932. Additionally, it originated from the Greek word 'IOKONOMIA', meaning Household Management.


In [ ]:
while True:

    question = input("Ask a question (type exit to quit): ")

    if question.lower() == "exit":
        break

    answer = ask_question(
        question,
        vectorstore
    )

    print("\nAnswer:")
    print(answer)
    print("-" * 50)

Ask a question (type exit to quit): What is Scarcity

Answer:
Scarcity means that resources are not available in the required quantity to satisfy all the wants and needs.
--------------------------------------------------
Ask a question (type exit to quit): What is Demand

Answer:
Demand is the desire backed by the ability and willingness to pay for a commodity.
--------------------------------------------------
Ask a question (type exit to quit): What are the factors of production

Answer:
The factors of production are:

1. Land (La) - Surface soil + Natural resources
2. Labour (L) - Mentally and physically fit for work
3. Capital (K) - All man-made aids to production
4. Organization/Entrepreneurship - Combines all factors of production.
--------------------------------------------------
Ask a question (type exit to quit): exit
